## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## T1 Document classification Evaluation

- Macro Agent (4-ways: Med, Fin, Id, Not_for_underwriting): 1,173 tokens
- Fin Specialist: 246 tokens
- Id Specialist: 609 tokens
- Med Specialist: 612 tokens

In [ ]:
# ==========================================
# v13 CHANGELOG (vs v12) -- REVERT TWO DECISIONS BEFORE THEY SHIP
# ==========================================
# 1. All receipts (medical AND non-medical) now route to financial_receipt. Removed
#    medical_receipt entirely -- reverts the edit to AGENT3_SYS_PROMPT, so that prompt
#    is back to exactly what your senior gave you, no sync needed.
# 2. Removed not_for_underwriting_subtag. not_for_underwriting is now a single flat
#    terminal class -- no eform/correspondence/envelope/portrait/illegible breakdown.
#    The descriptive examples stay in the prompt (still useful for the model to
#    recognize the pattern), just not captured as a structured output field anymore.
# ==========================================


import math
import time
import random
import os
import csv
import threading
import pandas as pd
from typing import Literal, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from openai import AzureOpenAI, BadRequestError, RateLimitError, APIConnectionError, APITimeoutError

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================

class MacroClassifierOutput(BaseModel):
    chain_of_thought: str
    macro_class: Literal["medical", "financial", "identification", "not_for_underwriting"]


class FinancialSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank", "financial_companyregistration",
        "financial_selfincomedeclaration", "financial_receipt", "financial_others"
    ]


class IdentificationSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "id_driverlicense", "id_fatca", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_visastamp", "id_thaievisa", "id_passport",
        "id_statelessid", "id_thainationalid", "id_workpermit",
        "id_houseregistration", "id_marriagecertificate",  # still tentative, see prior note
        "id_birthcertificate",
        "id_others"
    ]


class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck",
        "medical_lab", "medical_others"
    ]

# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================

# --- STEP 1: MACRO CLASSIFIER (replaces the old binary triage + 4-way router) ---
AGENT1_SYS_PROMPT = """You are a Document Macro Classifier for an insurance underwriting pipeline. Classify raw OCR/markdown text into exactly one of: `medical`, `financial`, `identification`, `not_for_underwriting`. Treat the patterns below as supporting evidence, not an exact-string checklist -- read holistically. Ignore PII.

### Ignore Watermarks
Insurance disclaimers (+ date/time stamps) are never evidence of document type -- ignore them completely.

### 1. medical
Route here for genuine clinical content: clinical notes, lab metrics, health-check parameters, prescriptions, test ranges, normal/abnormal flags. Garbled OCR with technical-looking terms next to numbers/units/ranges still counts as low-confidence medical (flag the uncertainty).
- *EXCLUSION:* Do NOT route here for a hospital/pharmacy receipt or billing statement -- those are `financial` regardless of medical context. Do NOT route here for correspondence that merely discusses or requests medical information without containing the clinical data itself -- that is `not_for_underwriting`.

### 2. financial
- **Bank/account identity:** Account number + bank name/branch, or a native transaction ledger (dated money movements, ideally a markdown table).
- **Corporate registration:** Registration number + certification/signature block.
- **Self income declaration & Income Overrides:** The person reporting, clarifying, or declaring their own income ("รายได้" / income). 
  - *CRITICAL OVERRIDE:* If income ("รายได้") details are present, you MUST route to `financial`, even if the document looks like an insurance e-Form containing premium payment keywords ("ชำระเบี้ยประกัน") or company headers.
- **Receipts/Proof-of-payment:** Including hospital/pharmacy/medical treatment receipts (ค่ายา, ค่ารักษา, ค่าห้อง), government permit fee receipts, retail receipts, or any other billing statement. All receipts are financial, regardless of what the payment was for.
- **General agreements/contracts:** Lease, sale, or other business contracts.

### 3. identification
A genuine ID/travel/civil-registry document with its own native structure (not a field filled into someone else's form):
- **National/Stateless ID:** Issuing country name/code (e.g., "LAO PDR", "RDP LAO"), a genuine name+DOB or name+issue/expiry block. Look for "รับรองสำเนาถูกต้อง" (certified true copy) or `<figure>` tags wrapping personal-detail blocks. Stateless-ID text or garbled non-Thai script (e.g., Lao misread as Thai) combined with a country-code header also counts.
- **Passports/Visas:** MRZ lines (letters/digits + long "<" runs, e.g., "PA123<<<<<<<"), or immigration/visa keywords ("VISA", "VISACLASS", "IMMIGRATION", "DEPARTED", "ADMITTED", "ENTRY") near messy digits/dates.
- **Thai E-Visa:** The literal term "E-VISA" printed on the document is a strong, standalone anchor.
- **Tax Forms:** FATCA-related tax forms (W-9, W-4, W-8BEN) or similar withholding declarations.
- **Work Permits:** Labor authorization booklets, employer/employee details, type-of-work (ประเภทงาน), Department of Employment text. Includes renewals/amendments (รายการเปลี่ยนเพิ่มประเภทงาน).
- **Civil Registry:** Household registration (ทะเบียนบ้าน), Marriage certificate (ใบสำคัญการสมรส / ทะเบียนสมรส), Birth certificate (สูติบัตร).
- **Other IDs:** Student IDs, employee badges, or any physical identity card with a photo placeholder/ID number.
- *Reminder:* An ID document stamped with an insurance watermark/date is still `identification`.

### 4. not_for_underwriting
Default when nothing above applies. The underwriter will not use this document.
- **Insurance application/policy paperwork (eForm Trap):** Forms containing applicant fields (เบี้ยประกัน, policy details). 
  - *SPECIFIC ANCHORS:* Documents opening with "บันทึกคำชี้แจงเกี่ยวกับใบคำขอประกันภัย", consent letters, OR documents containing "ชำระเบี้ยประกัน" (premium payment) mixed with a company name/address in the PageHeader. 
  - *EXCEPTION:* If the document explicitly declares actual income details ("รายได้"), route to `financial` instead.
- **Litmus test for eForms:** Strip out the applicant's name/ID/policy number -- is any standalone financial, clinical, or identity statement left? If nothing remains but "applying for / updating a policy," it belongs here.
- **Correspondence:** Letters/memos between the insured and underwriter requesting documents (administrative communication).
- **Envelope/mailing metadata:** Sender name, policy codes (e.g., "T1348646"), addressing text.
- **Photos/Noise:** Portrait photos or ID placeholders without issuing text, or completely illegible noise.

In `chain_of_thought`: Name the pattern found. If `not_for_underwriting`, briefly state which specific pattern it resembles."""

AGENT1_USER_PROMPT = "Classify the following raw OCR text into medical, financial, identification, or not_for_underwriting:\n{ocr_text}"


# --- STEP 2a: FINANCIAL SPECIALIST ---
AGENT_FINANCIAL_SYS_PROMPT = """You are an expert Financial Document Specialist. You receive text already confirmed as financial. Perform granular classification into one of six leaf classes. Use markdown structure (tables, headers) where present. Do not process PII.

- financial_bankstatement: running transactional ledgers, money transfers, account balances, mobile banking markers -- ideally a markdown table of dated transaction rows.
- financial_bookbank: static account identity headers (bilingual accounts, branches, bank names) without a long transaction ledger.
- financial_companyregistration: formal commercial corporate registry verbiage and certification signatures.
- financial_selfincomedeclaration: the person reporting, clarifying, or declaring their own income, salary, or source of income -- regardless of whether it's laid out as a form or questionnaire.
- financial_receipt: any proof-of-payment document -- hospital/pharmacy/medical treatment receipts, government license/permit fee receipts, retail receipts, or any other billing statement, regardless of what the payment was for.
- financial_others: financial in nature but doesn't cleanly match the above -- also covers general agreements/contracts (lease, sale, business contracts).

Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""

AGENT_FINANCIAL_USER_PROMPT = "Perform deep financial classification on this verified text:\n{ocr_text}"


# --- STEP 2b: IDENTIFICATION SPECIALIST ---
AGENT_ID_SYS_PROMPT = """You are an expert Identification Document Specialist. You receive text already confirmed as an identification/travel document. Perform granular classification. Use markdown structure where present. Do not process PII.
 
Reminder: insurance watermarks/disclaimers (+ date/time stamps) are never evidence of document type -- ignore them when classifying.
 
- id_thainationalid / id_foreigner_nationalid: national demographic card headers and identity issuing text blocks. Foreign-script documents (e.g. a Lao national ID) still count even if largely unreadable by OCR -- look for legible fragments like issuing country name/code, name, or a DOB pattern.
- id_passport: the passport bio data page -- photo/data block, passport number, nationality, an MRZ line (letters/digits + long "<" runs).
- id_visastamp: a Thai visa page and/or immigration control stamp -- visa category codes, "DEPARTED"/"ADMITTED"/"ENTRY" keywords, and (often garbled, since stamp text is curved) date-like digit strings near them. Does NOT include a Thai E-Visa (see id_thaievisa below) -- that's a distinct electronically-issued document, not a physical stamp/page in a passport.
- id_thaievisa: a Thai Electronic Visa (E-Visa) -- look for the literal term "E-VISA" printed on the document; this alone is a reliable, standalone anchor for this class.
- id_workpermit: labor authorization booklets and official employment permission context. This includes work permit RENEWALS and amendment/endorsement documents such as "Change or addition of category of work" (รายการเปลี่ยนเพิ่มประเภทงาน) -- these are the same underlying document type as the original permit, just an update to it.
- id_fatca: any FATCA-related US tax form -- W-9, W-4, W-8BEN, or similar FATCA/tax-status declaration or backup-withholding form, entity/individual tax certification sections.
- id_foreignerconfirmationform: an official government-issued confirmation/declaration form attesting to a foreign national's identity or status -- issued BY a government/authority ABOUT the person, not filled in by an agent for a policy application.
- id_statelessid: a stateless-person identity card -- card layout similar to a national ID (photo placeholder, ID number, name, DOB) but with issuing text indicating stateless/no-registered-nationality status.
- id_driverlicense: driver's license.
- id_houseregistration: household registration document (ทะเบียนบ้าน) -- household registration book header, house/address registry formatting, list of household members.
- id_marriagecertificate: marriage certificate (ใบทะเบียนสมรส) -- marriage registration terminology, groom/bride names, registrar signature, marriage date.
- id_birthcertificate: birth certificate (สูติบัตร) -- birth registration terminology, parents' names, date/place of birth, registrar signature/seal.
- id_others: clearly an ID-type artifact that doesn't cleanly match any subtype above -- e.g. a student ID card, employee badge, or other institutional ID card. Expected, valid outcome for these and other borderline documents -- don't force a wrong specific subtype. NOTE: audit this bucket periodically -- some things that land here (e.g. a license/permit renewal fee) may actually be a mislabeled receipt (financial_receipt) rather than a true identification document; if the content is fundamentally about a payment/fee rather than proving identity, flag this in chain_of_thought instead of forcing it here.
 
Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""
 

AGENT_ID_USER_PROMPT = "Perform deep identification classification on this verified text:\n{ocr_text}"


# --- MEDICAL SPECIALIST PROMPTS (UNCHANGED -- owned by a different prompt owner) ---
AGENT3_SYS_PROMPT = """You are a strict Medical Document Specialist. Categorize pre-verified medical OCR text into the exact clinical subclass. Focus on clinical structures and specific medical anchors.
 
### Subclass Priority & Evaluation Guides
Evaluate in this priority order -- if multiple patterns are present on the same page, the higher-priority match wins.
 
- **medical_lab (highest priority):** Laboratory test results. Contains tabular test data with columns for Test Name, Result, Units, and Reference Range.
  Keywords: CBC, BUN, Creatinine, Lipid Profile, Glucose, mg/dL, mmol/L, x10^3/uL.
  NOT included: waveform studies, signal recordings, or functional diagnostics (e.g., ECG, EKG, EMG, EEG, spirometry graphs, imaging reports) -- these go to medical_clinical instead (see below), unless the same page also carries extractable lab values, in which case medical_lab still wins.
 
- **medical_healthcheck (second priority):** Health check / physical examination data. Any page containing extractable vital signs or physical examination findings -- Blood Pressure, BMI, Heart Rate, Weight, Height, or a structured wellness summary. Applies regardless of whether the source document is a dedicated checkup report (e.g., "Annual Health Checkup", "Executive Health Screening") or a clinical encounter record (OPD/IPD) that includes a Physical Examination (PE) section with structured vitals.
 
- **medical_clinical (third priority):** Everything genuinely medical that isn't extractable lab data or extractable vitals. This is deliberately a broad category: medical_lab and medical_healthcheck feed a downstream extraction node that pulls structured values (test results, BMI, BP, weight) -- medical_clinical is the landing spot for real clinical content that node doesn't need structured values from. Includes:
  - Clinical narrative/notes: doctor consultation notes, progress notes, admission or discharge summaries, diagnosis lists, prescriptions, treatment plans, operative notes, hospital course documentation. Typically has visit/admission dates, narrative assessments, plans, and medication orders.
  - Pathology reports.
  - Imaging or functional diagnostic reports with no extractable lab or vitals data on the page -- X-ray, ultrasound, ECG/EKG, EMG, EEG, spirometry, or similar studies.
  NOT included: pages whose primary content is a table of vital signs or PE measurements (-> medical_healthcheck) or a lab results table (-> medical_lab).
 
- **medical_others:** Medical documents that don't structurally fit any category above.
  SPECIFIC INCLUSION: always route Sleep Test reports here.
 
In chain_of_thought: state the specific clinical anchors found and explicitly name the priority logic applied -- e.g. "Found lab table with mg/dL, overriding vitals", "Found BP/BMI but no labs", "X-ray report, no lab/vitals values present -> clinical", "Sleep test mention -> others" -- before making your final selection."""
 
AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"

# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Whole-response confidence (legacy). Prefer calculate_field_confidence for
    anything you act on."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_field_confidence(response_logprobs, field_value: str) -> float:
    """Isolates and averages logprobs only for the tokens making up the classification
    enum value itself, rather than the whole JSON payload including chain_of_thought."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]

    running = ""
    span_start, span_end = None, None
    for i, tok in enumerate(token_strs):
        prev_len = len(running)
        running += tok
        if field_value in running[max(0, prev_len - len(field_value)):]:
            end_idx = i
            partial = ""
            start_idx = i
            for j in range(i, -1, -1):
                partial = token_strs[j] + partial
                start_idx = j
                if field_value in partial:
                    break
            span_start, span_end = start_idx, end_idx

    if span_start is None:
        return calculate_confidence(response_logprobs)

    span_logprobs = [tokens[i].logprob for i in range(span_start, span_end + 1) if tokens[i].logprob is not None]
    if not span_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(span_logprobs) / len(span_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def call_with_retry(fn, *args, max_retries: int = 6, base_delay: float = 1.0,
                     max_delay: float = 60.0, **kwargs):
    """Wraps a single API call with retry + exponential backoff for 429s and transient
    connection errors. Respects Azure's Retry-After header when present."""
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            retry_after = None
            try:
                retry_after = float(e.response.headers.get("retry-after"))
            except Exception:
                pass
            delay = retry_after if retry_after is not None else min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, delay * 0.25)
            print(f"[RATE LIMIT] attempt {attempt + 1}/{max_retries}, sleeping {delay:.1f}s")
            time.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1)
            print(f"[TRANSIENT ERROR] {type(e).__name__}: {e} -- sleeping {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Exceeded max_retries ({max_retries}) calling {getattr(fn, '__name__', fn)}")


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI,
                          macro_model: str = "gpt-5-mini",
                          financial_model: str = "gpt-5",
                          id_model: str = "gpt-5",
                          med_model: str = "gpt-5") -> dict:
    """Executes the 2-stage architecture on a single piece of text.

    Step 1 (macro_model): single call, 4-way (medical / financial / identification /
        not_for_underwriting).
    Step 2 (financial_model / id_model / med_model): only called for medical / financial
        / identification. not_for_underwriting is terminal, flat, no subtag.
    """

    macro_response = call_with_retry(
        client.beta.chat.completions.parse,
        model=macro_model,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=MacroClassifierOutput,
        logprobs=True,
        top_logprobs=1
    )
    macro_data = macro_response.choices[0].message.parsed
    macro_class = macro_data.macro_class
    macro_conf = calculate_field_confidence(macro_response.choices[0].logprobs, macro_class)

    base_result = {
        "macro_decision": macro_class,
        "macro_reason": macro_data.chain_of_thought,
        "macro_confidence": macro_conf,
    }

    if macro_class == "medical":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=med_model,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    if macro_class == "financial":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=financial_model,
            messages=[
                {"role": "system", "content": AGENT_FINANCIAL_SYS_PROMPT},
                {"role": "user", "content": AGENT_FINANCIAL_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=FinancialSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    if macro_class == "identification":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=id_model,
            messages=[
                {"role": "system", "content": AGENT_ID_SYS_PROMPT},
                {"role": "user", "content": AGENT_ID_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=IdentificationSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {**base_result, "final_subcategory": spec_data.subcategory,
                "specialist_reason": spec_data.chain_of_thought, "specialist_confidence": spec_conf}

    # ---- Terminal: not_for_underwriting -- no Step 2 call ----
    return {
        **base_result,
        "final_subcategory": "not_for_underwriting",
        "specialist_reason": "Terminal at the macro classifier.",
        "specialist_confidence": macro_conf,
    }


# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================

_checkpoint_lock = threading.Lock()


def _process_one_row(idx, row, client, macro_model, financial_model, id_model, med_model):
    ocr_text = str(row['ocr_text'])

    try:
        res = run_cascade_pipeline(ocr_text, client, macro_model, financial_model, id_model, med_model)
    except BadRequestError as e:
        error_message = str(e).lower()
        if "content management policy" in error_message or "jailbreak" in error_message:
            print(f"[WARNING] Doc {idx + 1} blocked by Azure Jailbreak Filter. Skipping...")
            res = {
                "macro_decision": "blocked_by_firewall",
                "macro_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
                "macro_confidence": 0.0,
                "final_subcategory": "error_content_filter",
                "specialist_reason": "Skipped due to API security policy.",
                "specialist_confidence": 0.0
            }
        else:
            raise e
    except Exception as e:
        print(f"[ERROR] Doc {idx + 1} failed due to unexpected error: {e}")
        res = {
            "macro_decision": "api_error",
            "macro_reason": str(e),
            "macro_confidence": 0.0,
            "final_subcategory": "api_error",
            "specialist_reason": "API disconnected or failed.",
            "specialist_confidence": 0.0
        }

    combined_row = {
        "filename": row.get('filename'),
        "input_category_label": row.get('category'),
        "input_subcategory_label": row.get('subcategory'),
        **res
    }
    return idx, combined_row


def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         macro_model: str = "gpt-5-mini",
                         financial_model: str = "gpt-5",
                         id_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.

    macro_model / financial_model / id_model / med_model: deployment names for each
        stage -- must match your Azure resource's deployment names.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines.
    checkpoint_path: results are appended here as each document finishes.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """

    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )

    df = pd.read_csv(csv_path)

    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")

    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]

    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")

    fieldnames = None

    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)

    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client,
                             macro_model, financial_model, id_model, med_model): idx
            for idx, row in pending
        }

        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)

        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Macro: {combined_row['macro_decision']} -> Leaf: {combined_row['final_subcategory']}")

    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")

    output_df = pd.read_csv(checkpoint_path)
    return output_df

# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     macro_model="gpt-5-mini",
#     financial_model="gpt-5",
#     id_model="gpt-5",
#     med_model="gpt-5",
#     max_workers=5,
# )

## T2 Medical Imaging Flag

In [ ]:
"""
medical_imaging_flags_agent.py
================================
Separate multimodal agent for tagging medical_lab / healthcheck / clinical documents.
Now includes VISION FALLBACK: automatically encodes the document image (from filepath) 
and sends it alongside the OCR text, allowing the model to visually read ECG graphs 
and X-Rays when OCR text is missing or sparse.

Run as a script:
    python medical_imaging_flags_agent.py results.csv --endpoint https://your-endpoint.openai.azure.com/ --model gpt-4o
"""

import math
import time
import random
import os
import csv
import threading
import argparse
import base64
import mimetypes
import pandas as pd
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from openai import AzureOpenAI, BadRequestError, RateLimitError, APIConnectionError, APITimeoutError

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ==========================================
# 1. SCHEMA
# ==========================================

class MedicalImagingFlagsOutput(BaseModel):
    chain_of_thought: str
    contains_xray: bool
    contains_ultrasound: bool
    contains_ecg: bool

# ==========================================
# 2. PROMPT
# ==========================================

AGENT_IMAGING_FLAGS_SYS_PROMPT = """You are a Medical Imaging & Diagnostics Tagging Agent. You receive BOTH the OCR text and the raw Document Image. Your job is multi-label tagging -- a document can contain zero, one, two, or all three of these report types together.

Set each flag to true only if there is genuine, specific evidence of that report type's actual findings -- not just a passing mention that the test was ordered.

- contains_xray: true if the text OR image shows actual X-ray findings/impressions (e.g. chest X-ray, radiographic findings, "CXR", lung fields, bone/joint imaging).
- contains_ultrasound: true if the text OR image shows actual ultrasound findings/impressions (e.g. abdominal ultrasound, echo findings, sonography report content).
- contains_ecg: true if the text OR image shows actual ECG/EKG findings. *CRITICAL:* If the image visually contains an electrocardiogram waveform grid (squiggly heart-rate lines on a grid), you MUST set this to true, even if the OCR text is empty.

These three flags are independent. A plain outpatient note with no imaging/ECG content at all should have all three false.

In chain_of_thought: name the specific evidence found (e.g., "Found visual ECG waveform graph in image" or "Found 'normal sinus rhythm' in text"). Explicitly note if a report type is only mentioned in passing (ordered/scheduled) -- that should NOT count as true."""

AGENT_IMAGING_FLAGS_USER_PROMPT = "Identify which of these report types are present with genuine findings in this medical document:\n{ocr_text}"

# ==========================================
# 3. HELPERS (IMAGE ENCODING & CONFIDENCE)
# ==========================================

def encode_image(image_path: str) -> Optional[str]:
    """Reads a local image file and converts it to a base64 Data URI for the Vision API."""
    if not image_path or not os.path.isfile(image_path):
        return None
    try:
        mime_type, _ = mimetypes.guess_type(image_path)
        mime_type = mime_type or 'image/jpeg'
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        return f"data:{mime_type};base64,{encoded_string}"
    except Exception as e:
        print(f"Failed to encode image {image_path}: {e}")
        return None

def calculate_confidence(response_logprobs) -> float:
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_key_confidence(response_logprobs, key_name: str) -> float:
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]
    search_key = f'"{key_name}"'

    running = ""
    key_end_idx = None
    for i, tok in enumerate(token_strs):
        running += tok
        if search_key in running and key_end_idx is None:
            key_end_idx = i
            break

    if key_end_idx is None:
        return calculate_confidence(response_logprobs)

    value_logprobs = []
    started = False
    for i in range(key_end_idx, len(tokens)):
        tok = token_strs[i]
        if not started:
            if ":" in tok:
                started = True
            continue
        if "," in tok or "}" in tok:
            break
        if tokens[i].logprob is not None:
            value_logprobs.append(tokens[i].logprob)

    if not value_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(value_logprobs) / len(value_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)

# ==========================================
# 4. CORE CALL + RETRY
# ==========================================

def call_with_retry(fn, *args, max_retries: int = 6, base_delay: float = 1.0,
                     max_delay: float = 60.0, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            retry_after = None
            try:
                retry_after = float(e.response.headers.get("retry-after"))
            except Exception:
                pass
            delay = retry_after if retry_after is not None else min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, delay * 0.25)
            print(f"[RATE LIMIT] attempt {attempt + 1}/{max_retries}, sleeping {delay:.1f}s")
            time.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1)
            print(f"[TRANSIENT ERROR] {type(e).__name__}: {e} -- sleeping {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Exceeded max_retries ({max_retries}) calling {getattr(fn, '__name__', fn)}")


def run_imaging_flags_agent(ocr_text: str, image_path: str, client: AzureOpenAI, model: str = "gpt-4o",
                             temperature: float = 0.0, seed: int = 42) -> dict:
    """Multimodal document call. Sends text and image to the LLM."""
    
    # 1. Build base text content
    user_content = [
        {"type": "text", "text": AGENT_IMAGING_FLAGS_USER_PROMPT.format(ocr_text=ocr_text)}
    ]
    
    # 2. Append the image if it exists on disk
    encoded_img = encode_image(image_path)
    if encoded_img:
        user_content.append({
            "type": "image_url",
            "image_url": {
                "url": encoded_img,
                "detail": "auto" # Let OpenAI decide resolution tier based on image size
            }
        })
    else:
        print(f"Warning: Could not load image for Vision API at {image_path}. Falling back to text-only.")

    response = call_with_retry(
        client.beta.chat.completions.parse,
        model=model,
        messages=[
            {"role": "system", "content": AGENT_IMAGING_FLAGS_SYS_PROMPT},
            {"role": "user", "content": user_content}
        ],
        response_format=MedicalImagingFlagsOutput,
        logprobs=True,
        top_logprobs=1,
        temperature=temperature,
        seed=seed,
    )
    data = response.choices[0].message.parsed
    logprobs = response.choices[0].logprobs

    return {
        "contains_xray": data.contains_xray,
        "contains_xray_confidence": calculate_key_confidence(logprobs, "contains_xray"),
        "contains_ultrasound": data.contains_ultrasound,
        "contains_ultrasound_confidence": calculate_key_confidence(logprobs, "contains_ultrasound"),
        "contains_ecg": data.contains_ecg,
        "contains_ecg_confidence": calculate_key_confidence(logprobs, "contains_ecg"),
        "flags_reason": data.chain_of_thought,
    }

# ==========================================
# 5. BATCH EXECUTION
# ==========================================

_checkpoint_lock = threading.Lock()


def _process_one_row(idx, row, client, model, temperature, seed):
    ocr_text = str(row.get("ocr_text", ""))
    filepath = str(row.get("filepath", ""))
    
    try:
        res = run_imaging_flags_agent(ocr_text, filepath, client, model, temperature, seed)
    except BadRequestError as e:
        error_message = str(e).lower()
        if "content management policy" in error_message or "jailbreak" in error_message:
            print(f"[WARNING] Doc {idx + 1} blocked by Azure Jailbreak Filter. Skipping...")
            res = {
                "contains_xray": None, "contains_xray_confidence": 0.0,
                "contains_ultrasound": None, "contains_ultrasound_confidence": 0.0,
                "contains_ecg": None, "contains_ecg_confidence": 0.0,
                "flags_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
            }
        else:
            raise e
    except Exception as e:
        print(f"[ERROR] Doc {idx + 1} failed due to unexpected error: {e}")
        res = {
            "contains_xray": None, "contains_xray_confidence": 0.0,
            "contains_ultrasound": None, "contains_ultrasound_confidence": 0.0,
            "contains_ecg": None, "contains_ecg_confidence": 0.0,
            "flags_reason": f"API error: {e}",
        }

    combined_row = {"filepath": filepath, **res}
    return idx, combined_row


def run_imaging_flags_batch(sample_path: str, endpoint: str, token_provider,
                             filter_col: str = "final_subcategory",
                             filter_values=("medical_lab", "medical_healthcheck", "medical_clinical"),
                             model: str = "gpt-4o",
                             temperature: float = 0.0,
                             seed: int = 42,
                             max_workers: int = 5,
                             checkpoint_path: str = "imaging_flags_results.csv",
                             resume: bool = True):
    
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        azure_ad_token_provider=token_provider,
        api_version="2024-08-01-preview",
    )

    df = pd.read_csv(sample_path)
    if filter_col not in df.columns:
        raise ValueError(f"'{filter_col}' not found in {sample_path}. Available columns: {list(df.columns)}")

    scoped = df[df[filter_col].isin(filter_values)].copy()
    print(f"Filtered to {len(scoped)} of {len(df)} row(s) where {filter_col} in {filter_values}.")

    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if "filepath" in prior.columns:
            already_done = set(prior["filepath"].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")

    pending = [
        (idx, row) for idx, row in scoped.iterrows()
        if str(row.get("filepath")) not in already_done
    ]
    print(f"Running MULTIMODAL imaging-flags agent on {len(pending)} document(s) with max_workers={max_workers}...")

    fieldnames = None

    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)

    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client, model, temperature, seed): idx
            for idx, row in pending
        }
        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)

        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filepath')} "
                      f"xray={combined_row['contains_xray']} us={combined_row['contains_ultrasound']} "
                      f"ecg={combined_row['contains_ecg']}")

    print(f"\nDone. {completed} document(s) processed this run. Results in '{checkpoint_path}'.")
    return pd.read_csv(checkpoint_path)


def main():
    parser = argparse.ArgumentParser(description="Run the medical imaging flags agent over a results CSV.")
    parser.add_argument("sample_path", help="Path to the CSV (e.g. your main pipeline's results CSV).")
    parser.add_argument("--endpoint", required=True, help="Azure OpenAI endpoint URL.")
    parser.add_argument("--model", default="gpt-4o")
    parser.add_argument("--filter-col", default="final_subcategory")
    parser.add_argument("--max-workers", type=int, default=5)
    parser.add_argument("--checkpoint-path", default="imaging_flags_results.csv")
    args = parser.parse_args()

    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    ) 

    run_imaging_flags_batch(
        args.sample_path, args.endpoint, token_provider,
        filter_col=args.filter_col, model=args.model,
        max_workers=args.max_workers, checkpoint_path=args.checkpoint_path,
    )


if __name__ == "__main__":
    main()